### Different Models access using Langchain

In [2]:
import os
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')
from langchain.chat_models import init_chat_model
model = init_chat_model('groq:llama-3.3-70b-versatile')
response = model.invoke('Hello what is the whether in hyderabad')
response.content


"Hello! Hyderabad, being a city in southern India, has a tropical wet and dry climate. The weather can vary depending on the time of year. Here's a general idea of what you can expect:\n\n**Current Weather:**\nI'm a large language model, I don't have real-time access to current weather conditions. However, I can suggest some ways for you to find out the current weather in Hyderabad:\n\n1. Check online weather websites like AccuWeather, Weather.com, or the Indian Meteorological Department (IMD) website.\n2. Download a weather app on your smartphone, such as Dark Sky or Weather Underground.\n3. Tune into local news or radio stations for weather updates.\n\n**Typical Weather Patterns:**\nHyderabad experiences three main seasons:\n\n1. **Summer (March to May):** Hot and dry, with temperatures often reaching 40°C (104°F) or higher.\n2. **Monsoon (June to September):** Warm and humid, with significant rainfall during the southwest monsoon season.\n3. **Winter (October to February):** Mild an

In [3]:
response = model.invoke('Yes i would like to know more about the whether in Hyd..')
response.content

"Hyderabad, the capital city of Telangana, India, has a unique climate. The weather in Hyderabad is generally warm and dry, with three main seasons: summer, monsoon, and winter.\n\n**Summer (March to May)**: During this time, the temperature can soar up to 40°C (104°F) and sometimes even higher. The heat is intense, and the humidity is relatively low.\n\n**Monsoon (June to September)**: Hyderabad experiences a significant amount of rainfall during the monsoon season, with most of the rainfall occurring between June and August. The temperature cools down during this time, and the humidity increases.\n\n**Winter (December to February)**: Winters in Hyderabad are mild, with temperatures ranging from 12°C (54°F) to 25°C (77°F). The humidity is relatively low, and the weather is generally pleasant.\n\nHere's a rough idea of what you can expect the weather to be like in Hyderabad throughout the year:\n\n* **January**: Cool and dry, with average highs around 25°C (77°F)\n* **February**: Mild 

In [ ]:
os.environ['GOOGLE_API_KEY'] = os.getenv('GOOGLE_API_KEY')
gemini_model = init_chat_model('gemini-3.5-flash',model_provider="google_genai")
response_gemini = gemini_model.invoke('hey how are you my name is anil')
response_gemini.text

"Hi Anil! It's great to meet you. I'm doing really well, thank you for asking! \n\nHow are you doing today? How can I help you out?"

In [5]:
import langchain
langchain.__version__

'1.3.13'

### otherways of integrating model without chat model

In [6]:
###accessing openai using package without chat_model
from langchain_openai import ChatOpenAI
model = ChatOpenAI(model='gpt-5.6')
resp = model.invoke('how are you openAI')
resp.content

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [ ]:
###accessing grok using package without chat_model
from langchain_groq import ChatGroq
model = ChatGroq(model='llama-3.3-70b-versatile')
resp = model.invoke('how are you Groq')
resp.content

"I'm just a language model, I don't have feelings or emotions like humans do, but I'm functioning properly and ready to assist you. However, I'm curious - who or what is Groq? Is that a character or a reference I should be familiar with?"

In [10]:
###accessing gemini using package without chat_model
from langchain_google_genai import ChatGoogleGenerativeAI
model = ChatGoogleGenerativeAI(model = 'gemini-3.5-flash')
resp = model.invoke('How are you gemini')
resp.content[0]['text']

"Hello! I'm doing well, thank you for asking. How are you doing today? How can I help you?"

## Batch and streaming

In [ ]:
for chunk in model.stream('How are you gemini') :
    print(chunk.text, end = '|', flush=True)


Hello! I'|m doing really well, thank you for asking. How are you doing today? 

How can I help you?|||

In [ ]:
for chunk in model.batch(['hello what is my name?','how are you doing','do you have emotions']):
    print(chunk.content)    

In [26]:
##making different kind of tools
import os
from langchain.chat_models import init_chat_model
from langchain.tools import tool
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')

@tool
def check_whether (city:str)->str :
    '''give the whether of a city'''
    return f'whether in {city} is cloudy and sunny...'

new_gemini_model = init_chat_model(model='groq:llama-3.3-70b-versatile')
model_with_tool = new_gemini_model.bind_tools([check_whether])
resp = model_with_tool.invoke('what is the whether like in Hyderabad?')
# print(resp)
for chunk in resp.tool_calls:
    print(f'Tool : {chunk['name']}' )
    print(f'Args: {chunk['args']}')
    print(f'ID: {chunk['id']}')

Tool : check_whether
Args: {'city': 'Hyderabad'}
ID: x8dkj158t


In [ ]:
##Tools execution Loops
message = [{"role":"user","content":"what is the whether in Boston?"}]
ai_msg = model_with_tool.invoke(message)
message.append(ai_msg)

for tool_call in ai_msg.tool_calls:
   tool_resp = check_whether.invoke(tool_call)
   message.append(tool_resp)

final_resp = model_with_tool.invoke(message) 
final_resp.content

'I only provide the function call to get the weather. The actual weather in Boston is provided by the function call result, which is cloudy and sunny.'

In [ ]:
from langchain.messages import SystemMessage,HumanMessage,AIMessage
s_message = SystemMessage('You are Programmer with 30 Years of Experience in AI')
ai_message = AIMessage('How can i help you ?')
h_message = HumanMessage('explain about rag in 10 lines')

res = model.invoke([s_message,ai_message,h_message])
print(res)

ChatGoogleGenerativeAIError: Error calling model 'gemini-3.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.5-flash\nPlease retry in 5.955492081s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTi